# HPC Lab — Experiment 9
## Introduction to GPU Programming with CUDA

**Name:** Aditi Mittal | **PRN:** 23610067 | **Batch:** B1

---
> **Before running:** Go to `Runtime → Change runtime type → Hardware accelerator → T4 GPU`

In [ ]:
# Verify GPU is available
!nvidia-smi

---
## Task 1 — GPU Info
Prints all key GPU properties using the CUDA runtime API.

In [ ]:
%%writefile task1_gpu_info.cu
#include <cuda_runtime.h>
#include <stdio.h>

int main() {
    int deviceCount = 0;
    cudaGetDeviceCount(&deviceCount);
    if (deviceCount == 0) { printf("No CUDA-capable GPU found.\n"); return 1; }
    printf("Number of CUDA Devices: %d\n\n", deviceCount);
    for (int i = 0; i < deviceCount; i++) {
        cudaDeviceProp prop;
        cudaGetDeviceProperties(&prop, i);
        printf("=== GPU %d ===\n", i);
        printf("  Name                      : %s\n",   prop.name);
        printf("  Total Global Memory       : %.0f MB\n", prop.totalGlobalMem / (1024.0 * 1024.0));
        printf("  Compute Capability        : %d.%d\n", prop.major, prop.minor);
        printf("  Number of SMs             : %d\n",   prop.multiProcessorCount);
        printf("  Max Threads per Block     : %d\n",   prop.maxThreadsPerBlock);
        printf("  Max Grid Dimensions       : (%d, %d, %d)\n",
               prop.maxGridSize[0], prop.maxGridSize[1], prop.maxGridSize[2]);
        printf("  Warp Size                 : %d\n",   prop.warpSize);
        printf("  Memory Bus Width          : %d bits\n", prop.memoryBusWidth);
        printf("  L2 Cache Size             : %d KB\n", prop.l2CacheSize / 1024);
        printf("  Shared Memory per Block   : %zu KB\n", prop.sharedMemPerBlock / 1024);
    }
    return 0;
}

In [ ]:
!nvcc -O2 -o task1 task1_gpu_info.cu && ./task1

---
## Task 2 — Hello from GPU
Each thread prints its global thread ID with block and thread indices.

In [ ]:
%%writefile task2_hello_gpu.cu
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void helloKernel() {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Hello from GPU Thread %d  (Block %d, Thread %d within block)\n",
           tid, blockIdx.x, threadIdx.x);
}

int main() {
    printf("=== Configuration 1: 1 Block, 8 Threads ===\n");
    helloKernel<<<1, 8>>>();
    cudaDeviceSynchronize();

    printf("\n=== Configuration 2: 1 Block, 16 Threads ===\n");
    helloKernel<<<1, 16>>>();
    cudaDeviceSynchronize();

    printf("\n=== Configuration 3: 2 Blocks, 8 Threads per Block ===\n");
    helloKernel<<<2, 8>>>();
    cudaDeviceSynchronize();

    printf("\n=== Configuration 4: 4 Blocks, 8 Threads per Block ===\n");
    helloKernel<<<4, 8>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
!nvcc -O2 -o task2 task2_hello_gpu.cu && ./task2

---
## Task 3 — Thread and Block IDs
Displays Block ID, Thread ID within Block, and Global Thread ID.

In [ ]:
%%writefile task3_thread_ids.cu
#include <cuda_runtime.h>
#include <stdio.h>

__global__ void showIDs() {
    int blockId  = blockIdx.x;
    int threadId = threadIdx.x;
    int globalId = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Block %d | Thread %d | Global ID %d\n", blockId, threadId, globalId);
}

int main() {
    printf("=== 2 Blocks x 4 Threads ===\n");
    printf("Formula: Global ID = blockIdx.x * blockDim.x + threadIdx.x\n\n");
    showIDs<<<2, 4>>>();
    cudaDeviceSynchronize();

    printf("\n=== 3 Blocks x 4 Threads ===\n");
    showIDs<<<3, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
!nvcc -O2 -o task3 task3_thread_ids.cu && ./task3

---
## Task 4 — Vector Addition: CPU vs CUDA
Compares sequential CPU and CUDA GPU vector addition on N = 1,000,000 elements.

In [ ]:
%%writefile task4_vector_add.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <math.h>

#define N       1000000
#define THREADS 256

void vectorAddCPU(float *a, float *b, float *c, int n) {
    for (int i = 0; i < n; i++) c[i] = a[i] + b[i];
}

__global__ void vectorAddGPU(float *a, float *b, float *c, int n) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < n) c[id] = a[id] + b[id];
}

int main() {
    size_t bytes = N * sizeof(float);
    float *h_a = (float*)malloc(bytes), *h_b = (float*)malloc(bytes);
    float *h_c_cpu = (float*)malloc(bytes), *h_c_gpu = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_a[i] = i * 0.5f; h_b[i] = (N - i) * 0.5f; }

    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    vectorAddCPU(h_a, h_b, h_c_cpu, N);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    double cpu_ms = (t1.tv_sec - t0.tv_sec)*1e3 + (t1.tv_nsec - t0.tv_nsec)/1e6;
    printf("=== Sequential CPU Vector Addition ===\n");
    printf("CPU Execution Time : %.3f ms\n\n", cpu_ms);

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes); cudaMalloc(&d_b, bytes); cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);
    int blocks = (N + THREADS - 1) / THREADS;

    cudaEvent_t ev_start, ev_stop;
    cudaEventCreate(&ev_start); cudaEventCreate(&ev_stop);
    cudaEventRecord(ev_start);
    vectorAddGPU<<<blocks, THREADS>>>(d_a, d_b, d_c, N);
    cudaEventRecord(ev_stop); cudaEventSynchronize(ev_stop);
    float gpu_ms = 0.0f;
    cudaEventElapsedTime(&gpu_ms, ev_start, ev_stop);
    cudaMemcpy(h_c_gpu, d_c, bytes, cudaMemcpyDeviceToHost);

    printf("=== CUDA GPU Vector Addition ===\n");
    printf("Blocks launched       : %d\n", blocks);
    printf("Threads per Block     : %d\n", THREADS);
    printf("GPU Kernel Time       : %.3f ms\n\n", gpu_ms);

    int errors = 0;
    for (int i = 0; i < N; i++)
        if (fabsf(h_c_cpu[i] - h_c_gpu[i]) > 1e-4f) errors++;
    printf("=== Verification ===\n");
    printf("CPU Result[0] = %.2f\n", h_c_cpu[0]);
    printf("GPU Result[0] = %.2f\n", h_c_gpu[0]);
    printf("Errors        : %d\n", errors);
    printf("Status        : %s\n", errors == 0 ? "PASSED" : "FAILED");
    printf("Speedup       : %.2fx\n", cpu_ms / gpu_ms);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c_cpu); free(h_c_gpu);
    return 0;
}

In [ ]:
!nvcc -O2 -o task4 task4_vector_add.cu -lm && ./task4

---
## Task 5 — CUDA Memory Management
Demonstrates `cudaMalloc`, `cudaMemcpy`, `cudaFree` with transfer timing.

In [ ]:
%%writefile task5_memory_mgmt.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define N       500000
#define THREADS 256

__global__ void vectorAddKernel(float *a, float *b, float *c, int n) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < n) c[id] = a[id] + b[id];
}

int main() {
    size_t bytes = N * sizeof(float);
    printf("===== CUDA Memory Management Demo =====\n\n");

    printf("[HOST] Allocating host memory...\n");
    float *h_a = (float*)malloc(bytes), *h_b = (float*)malloc(bytes), *h_c = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_a[i] = 1.0f + i*0.001f; h_b[i] = 2.0f + i*0.001f; }
    printf("[HOST] Input arrays initialised.\n\n");

    printf("[DEVICE] Allocating device memory with cudaMalloc()...\n");
    float *d_a, *d_b, *d_c;
    cudaMalloc((void**)&d_a, bytes); cudaMalloc((void**)&d_b, bytes); cudaMalloc((void**)&d_c, bytes);
    printf("[DEVICE] Device memory allocated: 3 x %.1f MB\n\n", bytes/1024.0/1024.0);

    cudaEvent_t t0, t1;
    cudaEventCreate(&t0); cudaEventCreate(&t1);

    printf("[TRANSFER] Host -> Device...\n");
    cudaEventRecord(t0);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float h2d_ms; cudaEventElapsedTime(&h2d_ms, t0, t1);
    printf("[TRANSFER] H->D complete: %.3f ms\n\n", h2d_ms);

    int blocks = (N + THREADS - 1) / THREADS;
    printf("[GPU] Launching kernel: %d blocks x %d threads...\n", blocks, THREADS);
    cudaEventRecord(t0);
    vectorAddKernel<<<blocks, THREADS>>>(d_a, d_b, d_c, N);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float kern_ms; cudaEventElapsedTime(&kern_ms, t0, t1);
    printf("[GPU] Kernel complete: %.3f ms\n\n", kern_ms);

    printf("[TRANSFER] Device -> Host...\n");
    cudaEventRecord(t0);
    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);
    cudaEventRecord(t1); cudaEventSynchronize(t1);
    float d2h_ms; cudaEventElapsedTime(&d2h_ms, t0, t1);
    printf("[TRANSFER] D->H complete: %.3f ms\n\n", d2h_ms);

    int errors = 0;
    for (int i = 0; i < N; i++)
        if (fabsf(h_c[i] - (h_a[i]+h_b[i])) > 1e-4f) errors++;
    printf("[VERIFY] Errors: %d | Status: %s\n\n", errors, errors==0?"PASSED":"FAILED");

    printf("===== Timing Summary =====\n");
    printf("  Host -> Device transfer : %.3f ms\n", h2d_ms);
    printf("  Kernel execution        : %.3f ms\n", kern_ms);
    printf("  Device -> Host transfer : %.3f ms\n", d2h_ms);
    printf("  Total GPU pipeline      : %.3f ms\n", h2d_ms+kern_ms+d2h_ms);

    printf("\n[CLEANUP] Freeing device memory with cudaFree()...\n");
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    printf("[CLEANUP] Done.\n");
    return 0;
}

In [ ]:
!nvcc -O2 -o task5 task5_memory_mgmt.cu -lm && ./task5

---
## Task 6 — Block Size Benchmark
Tests thread counts 64, 128, 256, 512 and measures kernel execution time for vector addition.

In [ ]:
%%writefile task6_block_configs.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define N 1000000

__global__ void vectorAdd(float *a, float *b, float *c, int n) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < n) c[id] = a[id] + b[id];
}

float benchmarkKernel(float *d_a, float *d_b, float *d_c, int n, int threadsPerBlock) {
    int blocks = (n + threadsPerBlock - 1) / threadsPerBlock;
    vectorAdd<<<blocks, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaDeviceSynchronize();
    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);
    vectorAdd<<<blocks, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaEventRecord(stop); cudaEventSynchronize(stop);
    float ms = 0.0f;
    cudaEventElapsedTime(&ms, start, stop);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return ms;
}

int main() {
    size_t bytes = N * sizeof(float);
    float *h_a = (float*)malloc(bytes), *h_b = (float*)malloc(bytes);
    for (int i = 0; i < N; i++) { h_a[i] = (float)i; h_b[i] = (float)(N-i); }
    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes); cudaMalloc(&d_b, bytes); cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    int configs[] = {64, 128, 256, 512};
    printf("=== Vector Addition — Block Size Benchmark ===\n");
    printf("Vector Size: %d\n\n", N);
    printf("%-20s %-15s %-15s\n", "Threads/Block", "Blocks", "Kernel Time (ms)");
    printf("%-20s %-15s %-15s\n", "-------------", "------", "----------------");
    for (int i = 0; i < 4; i++) {
        int t = configs[i], b = (N+t-1)/t;
        float ms = benchmarkKernel(d_a, d_b, d_c, N, t);
        printf("%-20d %-15d %-15.3f\n", t, b, ms);
    }
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b);
    return 0;
}

In [ ]:
!nvcc -O2 -o task6 task6_block_configs.cu -lm && ./task6

---
## Task 7 — Sequential vs OpenMP vs CUDA
Compares three parallelism strategies for vector addition at multiple sizes.

In [ ]:
%%writefile task7_comparison.cu
#include <cuda_runtime.h>
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define THREADS_PER_BLOCK 256

void addSeq(float *a, float *b, float *c, long n) {
    for (long i = 0; i < n; i++) c[i] = a[i] + b[i];
}

void addOMP(float *a, float *b, float *c, long n) {
    #pragma omp parallel for schedule(static)
    for (long i = 0; i < n; i++) c[i] = a[i] + b[i];
}

__global__ void addCUDA(float *a, float *b, float *c, long n) {
    long id = (long)blockIdx.x * blockDim.x + threadIdx.x;
    if (id < n) c[id] = a[id] + b[id];
}

double getTimeMs() {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec * 1e3 + ts.tv_nsec / 1e6;
}

void runBenchmark(long n) {
    size_t bytes = n * sizeof(float);
    float *h_a=(float*)malloc(bytes), *h_b=(float*)malloc(bytes);
    float *h_seq=(float*)malloc(bytes), *h_omp=(float*)malloc(bytes), *h_gpu=(float*)malloc(bytes);
    for (long i=0; i<n; i++) { h_a[i]=i*0.1f; h_b[i]=(n-i)*0.1f; }

    double t0 = getTimeMs(); addSeq(h_a, h_b, h_seq, n); double seq_ms = getTimeMs()-t0;
    t0 = getTimeMs(); addOMP(h_a, h_b, h_omp, n); double omp_ms = getTimeMs()-t0;

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes); cudaMalloc(&d_b, bytes); cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);
    long blocks = (n + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    cudaEvent_t ev0, ev1;
    cudaEventCreate(&ev0); cudaEventCreate(&ev1);
    cudaEventRecord(ev0);
    addCUDA<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, n);
    cudaEventRecord(ev1); cudaEventSynchronize(ev1);
    float cuda_ms; cudaEventElapsedTime(&cuda_ms, ev0, ev1);
    cudaMemcpy(h_gpu, d_c, bytes, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (long i=0; i<n; i++) if (fabsf(h_seq[i]-h_gpu[i])>1e-3f) errors++;
    printf("  N = %8ld | Seq: %8.3f ms | OMP: %8.3f ms | CUDA: %8.3f ms | Err: %d\n",
           n, seq_ms, omp_ms, (double)cuda_ms, errors);

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_seq); free(h_omp); free(h_gpu);
}

int main() {
    printf("=== Sequential vs OpenMP vs CUDA — Vector Addition ===\n");
    printf("OpenMP threads: %d\n\n", omp_get_max_threads());
    long sizes[] = {100000L, 1000000L, 10000000L, 100000000L};
    for (int i=0; i<4; i++) runBenchmark(sizes[i]);
    return 0;
}

In [ ]:
!nvcc -O2 -Xcompiler -fopenmp -o task7 task7_comparison.cu -lgomp -lm && ./task7

---
## Task 8 — Array Squaring: CPU vs CUDA
Each GPU thread squares one element: `output[i] = input[i] * input[i]`

In [ ]:
%%writefile task8_array_squaring.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <time.h>

#define N       1000000
#define THREADS 256

void squareCPU(float *in, float *out, int n) {
    for (int i = 0; i < n; i++) out[i] = in[i] * in[i];
}

__global__ void squareGPU(float *in, float *out, int n) {
    int id = blockIdx.x * blockDim.x + threadIdx.x;
    if (id < n) out[id] = in[id] * in[id];
}

int main() {
    size_t bytes = N * sizeof(float);
    float *h_in=(float*)malloc(bytes), *h_cpu=(float*)malloc(bytes), *h_gpu=(float*)malloc(bytes);
    for (int i=0; i<N; i++) h_in[i] = i * 0.01f;

    struct timespec t0, t1;
    clock_gettime(CLOCK_MONOTONIC, &t0);
    squareCPU(h_in, h_cpu, N);
    clock_gettime(CLOCK_MONOTONIC, &t1);
    double cpu_ms = (t1.tv_sec-t0.tv_sec)*1e3 + (t1.tv_nsec-t0.tv_nsec)/1e6;

    float *d_in, *d_out;
    cudaMalloc(&d_in, bytes); cudaMalloc(&d_out, bytes);
    cudaMemcpy(d_in, h_in, bytes, cudaMemcpyHostToDevice);
    int blocks = (N + THREADS - 1) / THREADS;
    cudaEvent_t ev0, ev1;
    cudaEventCreate(&ev0); cudaEventCreate(&ev1);
    cudaEventRecord(ev0);
    squareGPU<<<blocks, THREADS>>>(d_in, d_out, N);
    cudaEventRecord(ev1); cudaEventSynchronize(ev1);
    float gpu_ms; cudaEventElapsedTime(&gpu_ms, ev0, ev1);
    cudaMemcpy(h_gpu, d_out, bytes, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int i=0; i<N; i++) if (fabsf(h_cpu[i]-h_gpu[i])>1e-3f) errors++;

    printf("=== Array Squaring: CPU vs CUDA ===\n");
    printf("Array Size          : %d\n", N);
    printf("Threads per Block   : %d\n", THREADS);
    printf("Blocks launched     : %d\n\n", blocks);
    printf("CPU Time            : %.3f ms\n", cpu_ms);
    printf("GPU Kernel Time     : %.3f ms\n", gpu_ms);
    printf("Speedup             : %.2fx\n\n", cpu_ms / gpu_ms);
    printf("=== Verification ===\n");
    printf("Errors              : %d\n", errors);
    printf("Status              : %s\n\n", errors == 0 ? "PASSED" : "FAILED");
    printf("=== Sample Values ===\n");
    printf("in[5]  = %.2f  =>  out[5]  = %.2f  (expected %.2f)\n", h_in[5], h_gpu[5], h_in[5]*h_in[5]);
    printf("in[10] = %.2f  =>  out[10] = %.2f  (expected %.2f)\n", h_in[10], h_gpu[10], h_in[10]*h_in[10]);
    printf("in[100]= %.2f  =>  out[100]= %.2f  (expected %.2f)\n", h_in[100], h_gpu[100], h_in[100]*h_in[100]);

    cudaFree(d_in); cudaFree(d_out);
    free(h_in); free(h_cpu); free(h_gpu);
    return 0;
}

In [ ]:
!nvcc -O2 -o task8 task8_array_squaring.cu -lm && ./task8